# Modelling the car

Here is one row of the starter's telemetry on `monza-like`, 150 s into the race, and the 20 actions the starter applied next.
**Where is the car one second later?**

That is the whole unit.
A controller that only reacts can fix an error after it has happened.
A controller that can **predict** sees the corner, the overshoot and the wall before the car gets there.
Units 6 (the speed a corner allows) and 10 (model predictive control) are built on exactly this question, so the model has to be right and its numerical error has to be known.

By the end of this lesson you will have a model of the car, two ways to integrate it, a measurement of each one's error and its *order*, and a `predict()` function in the submission file that reproduces racekit's next second to 1e-8 m.

One honest note first.
Like Unit 2, this unit does **not** change the driving.
`agents/unit-03.py` keeps the starter's control law exactly and races exactly like it.
It adds the model and a rollout; Units 4, 5 and 10 build controllers that use them.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from racekit.cars import KinematicCar
from racekit.runner import CONTROL_DT, load_submission, run_race, shim_installed
from racekit.tracks import load_track

STARTER = Path("../../projects/racekit/agents/unit-01.py")
UNIT_03 = Path("../../projects/racekit/agents/unit-03.py")

race = run_race(STARTER, "monza-like", car=KinematicCar(jitter=0.0))
telemetry = race.telemetry
print(race)

ROW = 3000
now = telemetry[ROW]
later = telemetry[ROW + 20]
print(f"t = {now['t']:.2f} s: x = {now['x']:.3f}, y = {now['y']:.3f}, yaw = {now['yaw']:.4f} rad, v = {now['v']:.4f} m/s")
print("the next 20 steers:", np.round(telemetry["steer"][ROW + 1:ROW + 21], 4))
print(f"t = {later['t']:.2f} s: x = {later['x']:.3f}, y = {later['y']:.3f}, yaw = {later['yaw']:.4f} rad  <- the answer")
assert race.finished and race.collisions == 0 and abs(race.total_time - 882.55) < 1e-9

Two things are already visible.
The yaw jumped from about −2.96 to +3.05 rad in one second: the car turned through about 0.27 rad, and racekit **wraps** the yaw to [−π, π), exactly as Unit 2's `wrap_angle` does.
Any comparison of headings must be wrapped.
And the speed did not move: 19.4933 m/s, to four decimals, a second apart.
§3 explains that number, and it is the one you met in Unit 1.

Row `i` of the telemetry holds the action applied **during** tick `i` and the state **after** it.
So the state at row `ROW + 20` is reached from the state at row `ROW` by the actions of rows `ROW + 1` … `ROW + 20`, each held for one 0.05 s tick.

## 1. The state and the question, precisely

The car's state is four numbers, **(x, y, ψ, v)**: the position of the rear axle in the world frame (metres), the heading ψ (radians, counter-clockwise from +x), and the speed v along the heading (m/s).
An action is (throttle, steer, brake).
A **model** is a function

$$\dot{\mathbf s} = f(\mathbf s, \mathbf u), \qquad \mathbf s = (x, y, \psi, v), \quad \mathbf u = (\text{throttle}, \text{steer}, \text{brake}).$$

Predicting one tick ahead means solving this ODE over 0.05 s with **u held constant**, because that is what the runner does: it applies your action and advances the world one tick.
Predicting one second ahead is 20 of those, chained.

Two separate things can go wrong, and this unit keeps them apart:

- the **model** can be wrong (f is not how the car moves);
- the **integration** can be wrong (f is right, but the numerical solution of the ODE is not).

On racekit's kinematic car the first error is zero by construction: we write down the same f that racekit uses.
That makes this the place to measure the second one cleanly.
§8 is about the first.

## 2. The kinematic bicycle and its circle

Collapse each axle to one wheel in the middle: a **bicycle**.
"Kinematic" means the wheels do not slip: each wheel moves only in the direction it points.
Take the rear axle as the reference point, with wheelbase L between the axles and the front wheel turned by δ.

The rear wheel moves along the heading, so

$$\dot x = v\cos\psi, \qquad \dot y = v\sin\psi.$$

Both wheels roll without slipping, so their velocities are perpendicular to the lines through each wheel's axle.
Those two lines meet at the **instantaneous centre of rotation**.
The rear axle line is perpendicular to the body; the front one is rotated by δ.
The triangle formed by the rear wheel, the front wheel and that centre has a right angle at the rear wheel, a side L along the body and an angle δ at the centre, so the rear wheel turns on a circle of radius

$$R = \frac{L}{\tan\delta}, \qquad \dot\psi = \frac{v}{R} = \frac{v\tan\delta}{L}.$$

Two conventions turn this into racekit's car:

- the steer command maps to the wheel angle as **δ = −steer · δ_max**, because a positive steer turns **right** (the official interface), while a positive δ turns left (counter-clockwise, ψ increasing);
- the signed curvature of the path is **κ = tan δ / L**, left positive. That is the same sign convention as Unit 2's Menger curvature of the waypoints.

racekit's kinematic car has L = 2.875 m and δ_max = 70°.
These are the **kinematic car's** parameters: a model, not the truth about CARLA's vehicle (§8).

In [ ]:
WHEELBASE = 2.875  # m, L
DELTA_MAX = math.radians(70.0)  # the wheel angle at |steer| = 1


def wheel_angle(steer):
    # A positive steer turns right; a positive wheel angle turns left.
    return -steer * DELTA_MAX


def path_curvature(steer):
    # Signed curvature kappa = tan(delta) / L, left positive (1/m).
    return math.tan(wheel_angle(steer)) / WHEELBASE


for steer in (0.05, 0.1, 0.2, 1.0):
    kappa = path_curvature(steer)
    print(f"steer {steer:4}: delta = {math.degrees(wheel_angle(steer)):6.2f} deg, kappa = {kappa:+.5f} 1/m, R = {1 / abs(kappa):7.3f} m (a right turn)")

RADII = {steer: 1.0 / abs(path_curvature(steer)) for steer in (0.05, 0.1, 0.2)}
assert abs(RADII[0.05] - 47.006) < 5e-4 and abs(RADII[0.1] - 23.415) < 5e-4 and abs(RADII[0.2] - 11.531) < 5e-4
assert path_curvature(0.1) < 0 < path_curvature(-0.1)

At full lock the car turns on a 1.05 m circle: a toy number, because no real car can take 70° of wheel angle at speed, which is one reason the model is not the truth.
The useful direction is the other one: **what steer does a corner need?**
Unit 2 gave you the curvature of the track from three waypoints.
Invert κ = tan(−steer · δ_max) / L:

$$\text{steer} = -\frac{\arctan(L\,\kappa)}{\delta_{\max}}.$$

Here it is for the tightest corner of each track, with Unit 2's signed Menger curvature of consecutive waypoint triples.
(`math.atan2(L * kappa, 1.0)` is arctan.)

In [ ]:
def menger_curvature(xs, ys):
    # Unit 2: the signed curvature at each waypoint from (i - 1, i, i + 1), cyclic; left positive.
    n = len(xs)
    before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
    ax, ay, cx, cy = xs[before], ys[before], xs[after], ys[after]
    twice_area = (xs - ax) * (cy - ay) - (ys - ay) * (cx - ax)
    sides = np.hypot(xs - ax, ys - ay) * np.hypot(cx - xs, cy - ys) * np.hypot(cx - ax, cy - ay)
    return 2.0 * twice_area / sides


STEER_FOR_TIGHTEST = {}
for name in ("oval", "test-circuit", "monza-like"):
    track = load_track(name)
    kappa = menger_curvature(track.x, track.y)
    tightest = int(np.argmax(np.abs(kappa)))
    steer_needed = -math.atan2(WHEELBASE * kappa[tightest], 1.0) / DELTA_MAX
    STEER_FOR_TIGHTEST[name] = steer_needed
    print(f"{name:>12}: R_min = {1 / abs(kappa[tightest]):6.2f} m, needs steer {steer_needed:+.4f} ({'left' if kappa[tightest] > 0 else 'right'})")
assert abs(abs(STEER_FOR_TIGHTEST["test-circuit"]) - 0.1296) < 5e-4

The tightest corner on any of the three tracks needs only about 13% of the steering range, in the kinematic model.
That is why the starter's steer stays small and why it never runs out of steering: on this car the corners are geometrically easy.
What they are *not* is fast; that limit (grip) arrives in Unit 6.

## 3. The longitudinal map and the terminal speed

The speed changes by three accelerations (m/s²):

$$\dot v = a_T\,\text{throttle} \;-\; a_B\,\text{brake}\,\operatorname{sign}(v) \;-\; c\,v\,|v|,$$

with a_T = 6, a_B = 8 and c = 4·10⁻⁴ m⁻¹ on racekit's kinematic car.
The brake acts **against the motion** (it is zero at v = 0, so braking never drives the car backwards), and the quadratic drag c·v·|v| is a lumped aerodynamic and rolling loss.
This is the **longitudinal map**: from pedal commands to acceleration.

Set v̇ = 0 to find where the speed settles.

- **Full throttle, no brake:** a_T = c·v², so the terminal speed is √(a_T / c) ≈ 122.5 m/s. The car never gets near it in a race.
- **The starter's speed controller:** throttle = 0.05·(20 − v), clipped to [0, 1]. Below 20 m/s the brake is 0, so the balance is
  $$a_T \cdot 0.05\,(20 - v) = c\,v^2 \quad\Longrightarrow\quad c\,v^2 + 0.3\,v - 6 = 0.$$

A proportional controller needs a nonzero error to produce the throttle that balances drag, so it settles **below** its target.
That is Unit 1's Exercise 6, now with the model behind it.

In [ ]:
A_THROTTLE = 6.0  # m/s^2 at full throttle
A_BRAKE = 8.0  # m/s^2 at full brake, against the motion
C_DRAG = 4.0e-4  # 1/m


def acceleration(v, throttle, brake):
    accel = A_THROTTLE * throttle - C_DRAG * v * abs(v)
    if v > 0.0:
        accel -= A_BRAKE * brake
    elif v < 0.0:
        accel += A_BRAKE * brake
    return accel


terminal_full = math.sqrt(A_THROTTLE / C_DRAG)
b = A_THROTTLE * 0.05
v_settle = (-b + math.sqrt(b * b + 4.0 * C_DRAG * A_THROTTLE * 0.05 * 20.0)) / (2.0 * C_DRAG)
cruise = float(np.median(telemetry["v"][telemetry["t"] > 60.0]))
print(f"terminal speed at full throttle: {terminal_full:.2f} m/s")
print(f"the starter's equilibrium (the quadratic): {v_settle:.4f} m/s")
print(f"the starter's median speed after 60 s on monza-like: {cruise:.4f} m/s")
print("acceleration there:", acceleration(v_settle, 0.05 * (20.0 - v_settle), 0.0))
assert abs(terminal_full - 122.47) < 0.005
assert abs(v_settle - 19.49) < 0.005 and abs(cruise - v_settle) < 1e-4

**19.49 m/s**: the number Unit 1 measured and called "a little below 20" is the root of a quadratic.
The deterministic starter sits on it, to 1e-4 m/s, for almost the entire race.
Keep that in mind for §7: on a straight at the equilibrium every derivative except the position's is zero.

Now the whole model in one function.

In [ ]:
def derivatives(state, action):
    # The kinematic bicycle with the longitudinal map: (x', y', psi', v').
    x, y, psi, v = state
    throttle, steer, brake = action
    return np.array([
        v * math.cos(psi),
        v * math.sin(psi),
        v * path_curvature(steer),
        acceleration(v, throttle, brake),
    ])


print(derivatives(np.array([0.0, 0.0, 0.0, 10.0]), (0.5, 0.1, 0.0)))

## 4. Euler, step by step

The simplest integrator follows the slope at the start of the step for the whole step:

$$\mathbf s_{n+1} = \mathbf s_n + h\,f(\mathbf s_n, \mathbf u).$$

Taylor-expand the true solution: s(t + h) = s + h·f + (h²/2)·f′ + O(h³).
Euler keeps the first two terms, so each step makes a **local error of order h²**.
Over a fixed horizon T there are T/h steps, so the **global error is of order h**: Euler is a *first-order* method.
Halve the step and you halve the error, and pay twice the work.

Watch what that means on the simplest manoeuvre there is: a steady circle.
Hold the steer and the drag-balancing throttle (c·v²/a_T, so v stays exactly 10 m/s), and step for 2000 ticks of h = 0.05 s.
The true path is the circle of §2: for a right turn from the origin heading +x, its centre is (0, −R).

In [ ]:
def euler_step(state, action, h):
    return state + h * derivatives(state, action)


def circle_distances(step, steer, h=0.05, steps=2000, v=10.0):
    # Hold steer at the drag-balancing throttle; return each step's distance from the true centre minus R.
    radius = 1.0 / abs(path_curvature(steer))
    action = (C_DRAG * v * v / A_THROTTLE, steer, 0.0)
    state = np.array([0.0, 0.0, 0.0, v])
    out = np.zeros(steps)
    for n in range(steps):
        state = step(state, action, h)
        out[n] = math.hypot(state[0], state[1] + radius) - radius
    return out


for steer, radius in RADII.items():
    deviation = circle_distances(euler_step, steer)
    first, last = np.max(np.abs(deviation[:200])), np.max(np.abs(deviation[-200:]))
    print(f"R = {radius:6.3f} m: Euler max |dist - R|, first 200 steps {first:.5f} m, last 200 steps {last:.5f} m (h v / 2 = 0.25)")
    assert 0.245 <= first <= 0.255 and 0.245 <= last <= 0.255
    assert abs(first - last) <= 1e-3

The error is **0.25 m at every radius, and it does not grow** over 2000 steps (about 3 to 14 laps).
That surprises people who expect Euler to "spiral out".
Here is why it does not.

With v and the steer held, every Euler step moves **h·v = 0.5 m** along the current heading and then turns by the same **h·v·κ**.
Equal sides, equal turns: the trajectory is an exact **regular polygon**.
Its vertices lie on a circle whose radius exceeds R by only about R·(hv/2R)²/6, a millimetre or less here.
But the polygon's first side starts at the origin *tangent* to the true circle, so the whole polygon is shifted forward along the initial heading by about **h·v/2**: Euler uses the heading at the start of each step, which is half a step *behind* the average heading over it.
A circle shifted by 0.25 m is between R − 0.25 and R + 0.25 from the true centre, once per lap.

So Euler's first-order error here is **a half-step lag of the whole circle**.
That is why the statistic is the *maximum* |dist − R| over a window: the signed deviation swings between about ±h·v/2 once per lap, so its mean says nothing.
Halve the step and the lag halves:

In [ ]:
deviation = circle_distances(euler_step, 0.1)
half_step = circle_distances(euler_step, 0.1, h=0.025, steps=4000)
print(f"h = 0.05:  max |dist - R| = {np.max(np.abs(deviation)):.5f} m")
print(f"h = 0.025: max |dist - R| = {np.max(np.abs(half_step)):.5f} m")
assert abs(np.max(np.abs(half_step)) - 0.125) < 0.002

t = 0.05 * np.arange(1, 2001)
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t, deviation, label="Euler, h = 0.05")
ax.plot(0.025 * np.arange(1, 4001), half_step, label="Euler, h = 0.025")
ax.axhline(0.25, color="grey", linestyle="--", linewidth=1.0)
ax.axhline(-0.25, color="grey", linestyle="--", linewidth=1.0)
ax.set_xlabel("t (s)")
ax.set_ylabel("dist - R (m)")
ax.set_title("Euler on a steady circle (R = 23.415 m, v = 10 m/s): a bounded half-step lag, not a drift")
ax.legend()
plt.show()

## 5. RK4

Euler samples the slope once, at the start.
The classical **Runge–Kutta** method samples it four times and averages:

$$\begin{aligned}
\mathbf k_1 &= f(\mathbf s_n), &
\mathbf k_2 &= f(\mathbf s_n + \tfrac h2 \mathbf k_1), \\
\mathbf k_3 &= f(\mathbf s_n + \tfrac h2 \mathbf k_2), &
\mathbf k_4 &= f(\mathbf s_n + h\,\mathbf k_3),
\end{aligned}
\qquad
\mathbf s_{n+1} = \mathbf s_n + \frac h6\left(\mathbf k_1 + 2\mathbf k_2 + 2\mathbf k_3 + \mathbf k_4\right).$$

k₁ is Euler's slope; k₂ and k₃ are two estimates at the midpoint; k₄ is at the end.
The weights 1, 2, 2, 1 are Simpson's rule, and the stages are chosen so the expansion of s(t + h) is matched through the h⁴ term.
The local error is O(h⁵), so RK4 is a **fourth-order** method: halve the step and the global error falls 16-fold.
It costs four evaluations of f per step instead of one.

racekit's own car is RK4 with the action held, in 10 substeps of 0.005 s per tick.

In [ ]:
def rk4_step(state, action, h):
    k1 = derivatives(state, action)
    k2 = derivatives(state + 0.5 * h * k1, action)
    k3 = derivatives(state + 0.5 * h * k2, action)
    k4 = derivatives(state + h * k3, action)
    return state + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


for steer, radius in RADII.items():
    worst = np.max(np.abs(circle_distances(rk4_step, steer)))
    print(f"R = {radius:6.3f} m: RK4 max |dist - R| over 2000 steps = {worst:.2e} m")
    assert worst < 1e-6

same_work = circle_distances(euler_step, 0.1, h=0.0125, steps=8000)
print(f"Euler with the same work (h = 0.0125): {np.max(np.abs(same_work)):.4f} m")

**RK4 stays on the circle to 3e-8 m** at the tightest of the three radii, for the same 0.05 s step that left Euler 0.25 m off.
Giving Euler RK4's budget (four steps of 0.0125 s per tick) only buys 0.0625 m.

## 6. The error orders, measured

"First order" and "fourth order" are claims about how the error scales with h.
Measure them.

The case: throttle 0.6, steer 0.15, from v₀ = 5 m/s, for T = 4 s: the car accelerates while turning right, so every state component changes.
The reference is RK4 at h = 10⁻⁴ s (its own error is far below anything measured here).
The error is the final position's distance from the reference, for h = 0.4, 0.2, 0.1 and 0.05 s.

If e(h) ≈ C·hᵖ, then e(h) / e(h/2) ≈ 2ᵖ, so the **observed order** is p ≈ log₂(e(h) / e(h/2)).
On a log–log plot the same law is a straight line of slope p.

In [ ]:
def rollout(step, h, horizon=4.0, action=(0.6, 0.15, 0.0), start=(0.0, 0.0, 0.0, 5.0)):
    state = np.array(start)
    for _ in range(int(round(horizon / h))):
        state = step(state, action, h)
    return state


STEPS = np.array([0.4, 0.2, 0.1, 0.05])
reference = rollout(rk4_step, 1e-4)
errors = {}
for name, step in (("Euler", euler_step), ("RK4", rk4_step)):
    errors[name] = np.array([math.hypot(*(rollout(step, h)[:2] - reference[:2])) for h in STEPS])
    ratios = errors[name][:-1] / errors[name][1:]
    print(f"{name:>5}: errors {[f'{e:.3g}' for e in errors[name]]} m; ratios {np.round(ratios, 2)}; orders {np.round(np.log2(ratios), 3)}")

euler_ratios = errors["Euler"][:-1] / errors["Euler"][1:]
rk4_ratios = errors["RK4"][:-1] / errors["RK4"][1:]
assert np.all((1.8 <= euler_ratios) & (euler_ratios <= 2.2))
assert np.all((14.0 <= rk4_ratios) & (rk4_ratios <= 18.0))
assert abs(errors["Euler"][-1] - 1.05) < 0.01

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(STEPS, errors["Euler"], "o-", label="Euler (slope 1)")
ax.loglog(STEPS, errors["RK4"], "s-", label="RK4 (slope 4)")
ax.set_xlabel("step h (s)")
ax.set_ylabel("position error after 4 s (m)")
ax.set_title("Convergence: error vs step size")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()

Euler's ratios are about 2 (order 1) and RK4's about 16 (order 4), exactly as the Taylor argument predicts.
Read the plot as an engineer:

- at the race's own tick (h = 0.05 s), Euler is **a metre off after 4 s**, RK4 is **75 nanometres off**;
- to get Euler to RK4's accuracy you would need about 10⁷ times more steps.

The orders are the asymptotic law; the ratios drift slightly from 2 and 16 at the largest h, where the higher-order terms still matter.
That is why the asserts allow a band, and why you always measure on a ladder of halvings rather than at one step.

## 7. Prediction from telemetry

Back to the hook.
Take the deterministic `monza-like` race, and from **180 start rows** (rows 200, 297, 394, … at a stride of 97) replay the **recorded** actions for 1 s (20 ticks), starting from the recorded state.
Compare the prediction with the recorded state 20 rows later.

In [ ]:
STARTS = np.arange(200, len(telemetry) - 20, 97)[:180]


def replay(step, substeps=1):
    # Predict 1 s ahead from each start row with the recorded actions; return the position errors (m).
    h = CONTROL_DT / substeps
    out = np.zeros(len(STARTS))
    for m, row in enumerate(STARTS):
        state = np.array([telemetry["x"][row], telemetry["y"][row], telemetry["yaw"][row], telemetry["v"][row]])
        for i in range(row + 1, row + 21):
            action = (telemetry["throttle"][i], telemetry["steer"][i], telemetry["brake"][i])
            for _ in range(substeps):
                state = step(state, action, h)
        end = row + 20
        out[m] = math.hypot(state[0] - telemetry["x"][end], state[1] - telemetry["y"][end])
    return out


replays = {
    "RK4, 1 step per tick": replay(rk4_step),
    "Euler, 1 step per tick": replay(euler_step),
    "Euler, 10 substeps": replay(euler_step, substeps=10),
}
for name, err in replays.items():
    print(f"{name:>24}: max {np.max(err):.3g} m, median {np.median(err):.3g} m")

print(len(STARTS), "start rows")
assert len(STARTS) == 180 and STARTS[0] == 200 and STARTS[1] == 297
assert np.max(replays["RK4, 1 step per tick"]) < 1e-6
euler_max = np.max(replays["Euler, 1 step per tick"])
euler10_max = np.max(replays["Euler, 10 substeps"])
assert abs(euler_max - 0.378) < 0.005 and abs(euler10_max - 0.0378) < 0.0005
assert 9.9 < euler_max / euler10_max < 10.1
assert np.median(replays["Euler, 1 step per tick"]) < 1e-5

Three results, each one a lesson.

- **RK4 with one step per tick reproduces racekit to about 2e-8 m.** racekit takes ten RK4 substeps per tick and we take one; the difference is RK4's own error at h = 0.05, which is tiny. Our model *is* the car here, so all that is left is integration error.
- **Euler is 0.378 m off at worst, and ten substeps cut that tenfold to 0.038 m.** Ten times smaller h, ten times smaller error: first-order convergence again, now on real telemetry. The worst windows are in the tight corners, where the heading changes fastest and Euler's half-step lag matters most.
- **Euler's *median* error is about 2e-6 m.** Most of a lap of `monza-like` is straight, and on a straight the starter sits at its **19.49 m/s drag equilibrium** (§3, Unit 1): v̇ = 0, ψ̇ ≈ 0, so f is constant along the path and Euler is exact. Integration error lives in the corners.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for name, err in replays.items():
    ax.semilogy(telemetry["t"][STARTS], err, ".", label=name)
ax.set_xlabel("start time (s)")
ax.set_ylabel("1 s position error (m)")
ax.set_title("Predicting 1 s ahead from telemetry, monza-like (zeros are not drawn on a log axis)")
ax.legend()
plt.show()

### Predicting without knowing the future actions

The replay cheated: it used the actions the starter *actually* applied over the next second.
A controller deciding now does not know them.
The simplest honest prediction holds the **current** action for the whole second.
This is measured here, not gated: it is a fact about the starter's steering, not about the integrator.

In [ ]:
held_errors = np.zeros(len(STARTS))
for m, row in enumerate(STARTS):
    state = np.array([telemetry["x"][row], telemetry["y"][row], telemetry["yaw"][row], telemetry["v"][row]])
    action = (telemetry["throttle"][row + 1], telemetry["steer"][row + 1], telemetry["brake"][row + 1])
    for _ in range(20):
        state = rk4_step(state, action, CONTROL_DT)
    end = row + 20
    held_errors[m] = math.hypot(state[0] - telemetry["x"][end], state[1] - telemetry["y"][end])
print(f"held current action, RK4: median {np.median(held_errors):.2g} m, 90th percentile {np.percentile(held_errors, 90):.3f} m, max {np.max(held_errors):.2f} m")

On the straights the held action is nearly perfect, but entering a corner the starter changes its steer within the second and the held prediction misses by up to about 2 m.
With RK4 the integration error is 10⁸ times smaller than that: **the uncertainty in a real prediction is the future actions (and, on CARLA, the model), not the integrator.**
Model predictive control (Unit 10) turns this around: it *chooses* the future actions, so they are known.

### The submission: `predict()`

`agents/unit-03.py` is the starter with Unit 2's instruments and this unit's model added.
Its `predict(state, actions, dt=0.05)` takes (x, y, ψ, v) and a sequence of (throttle, steer, brake), applies one RK4 step per action and returns the final state.
One exception: a tick in which the car can reach rest under the brake. There v' jumps at v = 0, RK4's smoothness assumption fails (one step across the jump can even predict the car reversing), so that tick is integrated exactly the way racekit's car integrates it (10 substeps, v set to 0 on a sign change or when the brake holds it at rest); the milestone checks that the two agree.
It is written with `math` scalars rather than NumPy arrays, because it runs inside `step()` every tick and a submission has a compute budget (Unit 1's `check_submission`: p99 ≤ 5 ms).
Each tick the file stores `self.predicted_state`, the held-action prediction 1 s ahead; nothing uses it for control yet.

Load the class and check it against the hook's row:

In [ ]:
with shim_installed():
    Agent, _ = load_submission(UNIT_03)
predict = Agent.predict  # the file's module-level predict(), exposed on the class

start = (now["x"], now["y"], now["yaw"], now["v"])
actions = [(telemetry["throttle"][i], telemetry["steer"][i], telemetry["brake"][i]) for i in range(ROW + 1, ROW + 21)]
x, y, psi, v = predict(start, actions)
print(f"predicted: x = {x:.6f}, y = {y:.6f}, psi = {psi:.6f} (unwrapped), v = {v:.6f}")
print(f"recorded:  x = {later['x']:.6f}, y = {later['y']:.6f}, yaw = {later['yaw']:.6f}, v = {later['v']:.6f}")
heading_gap = (psi - later["yaw"] + math.pi) % (2.0 * math.pi) - math.pi
print(f"position error {math.hypot(x - later['x'], y - later['y']):.2e} m, wrapped heading error {heading_gap:.2e} rad")
assert math.hypot(x - later["x"], y - later["y"]) < 1e-6 and abs(heading_gap) < 1e-9

`predict` does not wrap ψ: a rollout is a continuous integration, and an unwrapped heading is easier to difference.
The raw gap here is about 2π, which is exactly why a heading comparison is always wrapped.

The milestone checks the full state (x, y, ψ, v) on all 180 windows, and that the file still races exactly like the starter.

## 8. This is the *model* car

Everything above was exact because our f is racekit's f.
CARLA's vehicle is not a kinematic bicycle, and on race day the model error dominates:

- **tyres slip.** A real tyre makes lateral force by slipping at an angle; there is no slip in a kinematic model, so it has no grip limit and no understeer. The **dynamic bicycle** (Unit 6) adds slip and a friction circle, and with them the speed limit a corner imposes.
- **the parameters are guesses.** L = 2.875 m, δ_max = 70°, a_T = 6, a_B = 8 and c = 4·10⁻⁴ are racekit's kinematic car's numbers, not measurements of CARLA's car. Real throttle response depends on gear and engine speed, brakes are not linear in the pedal, and steering has a rate limit and a delay. **Unit 8 re-identifies these parameters from logged telemetry** with least squares, which is the honest way to get a model of a car you cannot open.
- **one tick is not instantaneous.** CARLA applies the control asynchronously to its physics, so "the action is held for exactly one tick" is itself an approximation.

None of that changes how this unit's integrators behave on a model like this one, because those are facts about solving a **smooth** ODE: Euler is first order, RK4 is fourth order, and on the race windows we measured, RK4's error at the 0.05 s tick is negligible next to the model's.
Two limits on that claim. The half-step lag is the steady circle's picture, not a law for every manoeuvre. And the orders assume f is smooth over the step: where it jumps (the brake at v = 0, a grip limit in Unit 6), a step that straddles the jump loses its order. The cure is to step *to* the jump (find when v reaches 0, then switch); substepping only shrinks the straddle, it does not remove it, and racekit's own car, which substeps, can creep at a few cm/s under full brake instead of stopping.
So the rule for the rest of the book: **integrate with RK4, and spend your doubt on the model.**

Next unit: PID control, the first controller that uses what Units 2 and 3 measure.